# 🤖 Knowledge Agent — RAG Ingestion Pipeline
> Fetches arXiv PDFs → deduplicates → converts via Docling GPU → pushes to Telegram & GitHub

In [ ]:
# ── Step 1: Set Tokens & Environment Variables ─────────────────
import os

# Read from Colab Secrets (Left sidebar -> 🔑 Key icon) or replace placeholders
try:
    from google.colab import userdata
    _get = userdata.get
except Exception:
    _get = lambda k: None

CONFIG = {
    "GITHUB_TOKEN": _get("GITHUB_TOKEN") or "YOUR_GITHUB_TOKEN",
    "GITHUB_REPO_URL": _get("GITHUB_REPO_URL") or "https://github.com/Nothing-dot-exe/arxiv-paper-scraper",
    "TELEGRAM_BOT_TOKEN": _get("TELEGRAM_BOT_TOKEN") or "YOUR_TELEGRAM_BOT_TOKEN",
    "TELEGRAM_GROUP_ID": _get("TELEGRAM_GROUP_ID") or "-1003958148223",
    "RAW_PDF_TOPIC_ID": "10",
    "TEXT_MD_TOPIC_ID": "11",
    "ADMIN_CHAT_ID": _get("ADMIN_CHAT_ID") or "YOUR_ADMIN_CHAT_ID",
    "PAPERS_PER_CATEGORY": "15"
}

for key, val in CONFIG.items():
    os.environ[key] = str(val)

print("✅ Credentials configured in environment!")


In [ ]:
# ── Step 2: Install Dependencies ──────────────────────────────────────────────
!pip install -q arxiv docling PyGithub requests thefuzz
print("✅ Dependencies installed successfully.")

In [ ]:
# ── Step 3: Setup Directories & Fetch Latest Script ─────────────────────
import os, urllib.request
from pathlib import Path

for d in ["markdown_files", "raw_pdfs", "logs"]:
    Path(d).mkdir(exist_ok=True)
    print(f"  📁 {d}/ ready")

print("Fetching latest knowledge_agent.py from GitHub repository...")
url = "https://raw.githubusercontent.com/Nothing-dot-exe/arxiv-paper-scraper/main/knowledge_agent.py"
headers = {}
if os.environ.get("GITHUB_TOKEN") and os.environ.get("GITHUB_TOKEN") != "YOUR_GITHUB_TOKEN":
    headers["Authorization"] = f"token {os.environ['GITHUB_TOKEN']}"
req = urllib.request.Request(url, headers=headers)
try:
    with urllib.request.urlopen(req) as resp, open("knowledge_agent.py", "wb") as f:
        f.write(resp.read())
    print("✅ Pipeline script verified and updated from GitHub!")
except Exception as e:
    print(f"⚠️ Remote fetch note ({e}). Using local/embedded knowledge_agent.py")


In [ ]:
# ── Step 4: Execute Pipeline ──────────────────────────────────────────────────
!python knowledge_agent.py